In [3]:
import sympy as sp
from sympy import (sqrt, Rational, symbols, simplify, together, fraction,
                   expand, collect, Poly, factor, solve, Eq, cancel)

a, aL = symbols("a alpha_L", positive=True)
t = symbols("t", positive=True)
s = symbols("s")
U = symbols("U", positive=True)
u = sqrt(t**2 + 4)
eps = symbols("epsilon", nonnegative=True)
x, sC, sT = symbols("x s_C s_T", positive=True)
half = Rational(1, 2)

# Setup
phi = (-a + sqrt(a) * sqrt(a + 4)) / 2
sC_0 = (1 - aL) * phi
sT_0 = aL * phi
D = a + phi
ss0 = {eps: 0, sC: sC_0, sT: sT_0}

hat_sT = solve(Eq(1 - x, (1 + eps)*x / (a + (1 + eps)*x)), x)
assert len(hat_sT) == 1
hat_sT = hat_sT[0]
GTE = (1 + eps)*hat_sT / (a + (1 + eps)*hat_sT) - phi / (a + phi)

D1 = a + sC + (1 + eps)*sT
D2 = a + sC + sT
F1 = (1 - aL - sC) - half*sC/D1 - half*sC/D2
F2 = (aL - sT) - half*(1 + eps)*sT/D1 - half*sT/D2
Q11, Q10 = half*(1 + eps)*sT/D1, half*sC/D1
Q01, Q00 = half*sT/D2, half*sC/D2
GTE_DD_est = (Q11 + Q01)/aL - (Q10 + Q00)/(1 - aL)
GTE_TSRN_est = Q11/(half*aL) - (Q01 + Q10 + Q00)/(1 - half*aL)

assert simplify(hat_sT.subs(eps, 0) - phi) == 0, "control steady state"
assert simplify(hat_sT - (-a + sqrt(a**2 + 4*a*(1 + eps))) / (2*(1 + eps))) == 0, "treatment steady state"
assert simplify(F1.subs(ss0)) == 0 and simplify(F2.subs(ss0)) == 0, "steady state at eps = 0"
assert simplify(GTE.subs(eps, 0)) == 0, "GTE(0)"
assert simplify(GTE_DD_est.subs(ss0)) == 0, "DD(0)"
assert simplify(GTE_TSRN_est.subs(ss0)) == 0, "TSRN(0)"
print("Setup: OK")

# Bias derivatives
A11 = -1 - (a + sT_0) / D**2
A12 = sC_0 / D**2
A21 = sT_0 / D**2
A22 = -1 - (a + sC_0) / D**2
det_J = A11*A22 - A12*A21
B1 = half * sC_0 * sT_0 / D**2
B2 = -half * sT_0 * (a + sC_0) / D**2
dsC = (-B1*A22 + B2*A12) / det_J
dsT = (-B2*A11 + B1*A21) / det_J
dGTE = 4 * a**Rational(3, 2) / (sqrt(a + 4) * (sqrt(a)*sqrt(a + 4) + a)**2)

B_DD = simplify((1/(1 - aL))*dsC - (1/aL)*dsT - dGTE)

dQ11 = half * (sT_0*(a + sC_0) + dsT*(a + sC_0) - sT_0*dsC) / D**2
dQ10 = half * (dsC*(a + sT_0) - sC_0*(sT_0 + dsT)) / D**2
dQ01 = half * (dsT*(a + sC_0) - sT_0*dsC) / D**2
dQ00 = half * (dsC*(a + sT_0) - sC_0*dsT) / D**2
B_TSRN = simplify(dQ11/(half*aL) - (dQ01 + dQ10 + dQ00)/(1 - half*aL) - dGTE)
Psi = simplify(B_DD + B_TSRN)

J = [[sp.diff(F1, sC), sp.diff(F1, sT)], [sp.diff(F2, sC), sp.diff(F2, sT)]]
for (i, j), Aij in {(0, 0): A11, (0, 1): A12, (1, 0): A21, (1, 1): A22}.items():
    assert simplify(J[i][j].subs(ss0) - Aij) == 0, "Jacobian"
assert simplify(det_J - (1 + (a + D)/D**2 + a/D**3)) == 0, "det J"
assert simplify(sp.diff(F1, eps).subs(ss0) - B1) == 0, "B1"
assert simplify(sp.diff(F2, eps).subs(ss0) - B2) == 0, "B2"
assert simplify(A11*dsC + A12*dsT + B1) == 0 and simplify(A21*dsC + A22*dsT + B2) == 0, "Cramer"
assert simplify(sp.diff(GTE, eps).subs(eps, 0) - dGTE) == 0, "dGTE"
assert cancel(GTE_DD_est - (sC/(1 - aL) - sT/aL) - (F1/(1 - aL) - F2/aL)) == 0, "DD equivalent form"
for Qk, dQk in ((Q11, dQ11), (Q10, dQ10), (Q01, dQ01), (Q00, dQ00)):
    chain = (sp.diff(Qk, eps) + sp.diff(Qk, sC)*dsC + sp.diff(Qk, sT)*dsT).subs(ss0)
    assert simplify(chain - dQk) == 0, "booking rate derivative"
print("Bias derivatives: OK")

# Theorem: B_DD < 0
B_DD_t = simplify(B_DD.subs(a, t**2))
assert aL not in B_DD_t.free_symbols, "B_DD depends on alpha_L"
num_DD, den_DD = fraction(together(B_DD_t))
N_DD = expand(collect(expand(num_DD), u)).subs(u, U)
D_DD = expand(collect(expand(den_DD), u)).subs(u, U)
assert all(c < 0 for k in (0, 1) for c in Poly(N_DD.coeff(U, k), t).coeffs()), "N(t) < 0"
assert all(c > 0 for k in (0, 1) for c in Poly(D_DD.coeff(U, k), t).coeffs()), "D(t) > 0"
assert expand(N_DD - (-t**2*(t**4 + 5*t**2 + 4 + U*(t**3 + 3*t)))) == 0, "N(t) factorization"

# Appendix C
assert expand(N_DD - (-t**6 - 5*t**4 - 4*t**2 - (t**5 + 3*t**3)*U)) == 0, "N(t) in paper"
d0 = 2*t**8 + 20*t**6 + 66*t**4 + 76*t**2 + 16
d1 = 2*t**7 + 16*t**5 + 38*t**3 + 24*t
assert expand(D_DD - (d0 + d1*U)) == 0, "D(t) in paper"
print("B_DD < 0: OK")

# Theorem: B_TSRN > 0
B_TSRN_t = simplify(B_TSRN.subs(a, t**2))
num_T, den_T = fraction(together(B_TSRN_t))
N_T = expand(collect(expand(num_T), u)).subs(u, U)
D_T = expand(collect(expand(den_T), u)).subs(u, U)
assert Poly(N_T, aL).degree() == 1 and Poly(D_T, aL).degree() == 1, "linear in alpha_L"
assert Poly(N_T, U).degree() == 1 and Poly(D_T, U).degree() == 1, "linear in u"

r0, r1 = [Poly(D_T.coeff(U, 0), aL).nth(k) for k in (0, 1)]
s0, s1 = [Poly(D_T.coeff(U, 1), aL).nth(k) for k in (0, 1)]
p0, p1 = [Poly(N_T.coeff(U, 0), aL).nth(k) for k in (0, 1)]
q0, q1 = [Poly(N_T.coeff(U, 1), aL).nth(k) for k in (0, 1)]
assert expand(N_T - (aL*(p1 + q1*U) + p0 + q0*U)) == 0, "numerator split"
assert expand(D_T - (aL*(r1 + s1*U) + r0 + s0*U)) == 0, "denominator split"

nonneg = lambda p: all(c >= 0 for c in Poly(p, t).all_coeffs())
nonpos = lambda p: all(c <= 0 for c in Poly(p, t).all_coeffs())
assert expand(r0 + 2*r1) == 0 and expand(s0 + 2*s1) == 0, "D = (alpha_L - 2)(r1 + s1 u)"
assert nonneg(r1) and nonneg(s1), "denominator sign"
assert nonneg(p1) and nonneg(q1) and expand(p1 + q1) != 0, "numerator slope"
assert nonpos(p1 + p0) and nonpos(q1 + q0) and expand(p1 + p0 + q1 + q0) != 0, "numerator at alpha_L = 1"

# Appendix C
paper = {
    "r1": (r1, t**16 + 18*t**14 + 133*t**12 + 518*t**10 + 1134*t**8 + 1372*t**6 + 833*t**4 + 198*t**2 + 8),
    "s1": (s1, t**15 + 16*t**13 + 103*t**11 + 340*t**9 + 606*t**7 + 560*t**5 + 231*t**3 + 28*t),
    "p1": (p1, 2*t**12 + 26*t**10 + 128*t**8 + 294*t**6 + 310*t**4 + 122*t**2 + 8),
    "q1": (q1, 2*t**11 + 22*t**9 + 88*t**7 + 154*t**5 + 110*t**3 + 22*t),
    "p0": (p0, -3*t**12 - 38*t**10 - 181*t**8 - 398*t**6 - 395*t**4 - 142*t**2 - 8),
    "q0": (q0, -3*t**11 - 32*t**9 - 123*t**7 - 204*t**5 - 135*t**3 - 24*t),
    "p1+p0": (p1 + p0, -t**12 - 12*t**10 - 53*t**8 - 104*t**6 - 85*t**4 - 20*t**2),
    "q1+q0": (q1 + q0, -t**11 - 10*t**9 - 35*t**7 - 50*t**5 - 25*t**3 - 2*t),
}
for name, (got, typed) in paper.items():
    assert expand(got - typed) == 0, name + " in paper"
print("B_TSRN > 0: OK")

# Boundary behavior of Psi
polys = {"N_DD": N_DD, "D_DD": D_DD, "N_T": N_T, "D_T": D_T}
for name, P in polys.items():
    assert P.coeff(U, 1).subs(t, 0) == 0, name
assert N_DD.subs(t, 0) == 0 and D_DD.subs(t, 0) == 16
assert expand(N_T.subs(t, 0) - 8*(aL - 1)) == 0 and expand(D_T.subs(t, 0) - 8*(aL - 2)) == 0
lim_0 = cancel(N_DD.subs(t, 0)/D_DD.subs(t, 0) + N_T.subs(t, 0)/D_T.subs(t, 0))
assert cancel(lim_0 - (aL - 1)/(aL - 2)) == 0, "limit at 0"

def lead(P):
    E, O = Poly(P.coeff(U, 0), t), Poly(P.coeff(U, 1), t)
    assert E.degree() == O.degree() + 1
    return E.degree(), factor(E.LC() + O.LC())

(dN, cN), (dD, cD) = lead(N_DD), lead(D_DD)
(dNT, cNT), (dDT, cDT) = lead(N_T), lead(D_T)
assert (dD - dN, dDT - dNT) == (2, 4)
assert (cN, cD) == (-2, 4)
assert expand(cNT - 2*(2*aL - 3)) == 0 and expand(cDT - 2*(aL - 2)) == 0
assert cancel(cNT/cDT - (2*aL - 3)/(aL - 2)) == 0, "limit at infinity"
print("Limits of Psi: OK")

# Unique threshold
red = lambda e: expand(sp.rem(expand(e), U**2 - t**2 - 4, U))
Num = red(N_DD*D_T + N_T*D_DD)
L_c, M_c = Num.coeff(U, 0), Num.coeff(U, 1)
parity = lambda p, k: all(m[0] % 2 == k for m in Poly(p, t).monoms())
for name, P in polys.items():
    assert parity(P.coeff(U, 0), 0) and parity(P.coeff(U, 1), 1), name + " parity"
assert parity(L_c, 0) and parity(M_c, 1)
assert Poly(L_c, aL).degree() == 1 and Poly(M_c, aL).degree() == 1
ell = expand(L_c).subs(t, sqrt(s))
mu = expand(cancel(M_c / t)).subs(t, sqrt(s))
assert ell.is_polynomial(s) and mu.is_polynomial(s)
R = expand(ell**2 - s*(s + 4)*mu**2)

A_q = (aL - 2)*(aL - 4)
B_q = -4*(2*aL**2 - 8*aL + 7)
C_q = 16*(aL - 1)**2
Q_q = A_q*s**2 + B_q*s + C_q
assert expand(R - 16*(s + 4)**3*Q_q) == 0, "R = 16(a+4)^3 Q"
assert all(r > 1 for r in solve(A_q, aL)) and A_q.subs(aL, 0) > 0
assert all(r > 1 for r in solve(2*aL**2 - 8*aL + 7, aL))
assert expand(Q_q.subs(s, 1) - (9*aL**2 - 6*aL - 4)) == 0
assert Q_q.subs({s: 1, aL: 0}) == -4 and Q_q.subs({s: 1, aL: 1}) == -1
assert expand(Q_q.subs(s, Rational(4, 3)) - Rational(64, 9)*(aL**2 - 1)) == 0
assert expand(Q_q.subs(s, 4)) == 32

Delta = expand(B_q**2 - 4*A_q*C_q)
assert expand(Delta - 16*(8*aL**2 - 24*aL + 17)) == 0, "discriminant"
assert all(r > 1 for r in solve(8*aL**2 - 24*aL + 17, aL))

# Check at a = 1, alpha_L = 1/2
at1 = {t: 1, U: sqrt(5), aL: half}
assert [e.subs(t, 1) for e in (p1, q1, p0, q0, r1, s1)] == [890, 398, -1165, -521, 4215, 1885]
assert expand(N_DD.subs(at1) - (-10 - 4*sqrt(5))) == 0
assert expand(D_DD.subs(at1) - (180 + 80*sqrt(5))) == 0
assert expand(N_T.subs(at1) - (-720 - 322*sqrt(5))) == 0
assert expand(D_T.subs(at1) + Rational(15, 2)*(843 + 377*sqrt(5))) == 0
assert expand((9 + 4*sqrt(5))*(9 - 4*sqrt(5))) == 1 and 843**2 - 5*377**2 == 4
B_DD_1 = sp.radsimp(N_DD.subs(at1) / D_DD.subs(at1))
B_TSRN_1 = sp.radsimp(N_T.subs(at1) / D_T.subs(at1))
assert simplify(B_DD_1 + (5 - 2*sqrt(5))/10) == 0
assert simplify(B_TSRN_1 - (3*sqrt(5) - 5)/15) == 0
Psi_1 = sp.radsimp(B_DD_1 + B_TSRN_1)
assert simplify(Psi_1 - (12*sqrt(5) - 25)/30) == 0 and (12*sqrt(5))**2 > 25**2
assert Q_q.subs({s: 1, aL: half}) < 0

a_star = simplify((-B_q + sqrt(Delta)) / (2*A_q))
assert simplify(a_star - (4*(2*aL**2 - 8*aL + 7) + 4*sqrt(8*aL**2 - 24*aL + 17))
                / (2*(aL - 2)*(aL - 4))) == 0, "closed form"
a_star_half = simplify(a_star.subs(aL, half))
assert simplify(a_star_half - (Rational(4, 3) + 8*sqrt(7)/21)) == 0
assert simplify(Psi.subs({a: a_star_half, aL: half})) == 0, "Psi(a*, 1/2) = 0"
print("Unique threshold: OK")
print("a* at alpha_L = 1/2:", a_star_half, "=", float(a_star_half))

# Monotonicity
dQ_daL = sp.diff(Q_q, aL)
assert expand(dQ_daL - 2*(s - 4)*((aL - 3)*s + 4*(1 - aL))) == 0
assert expand(sp.diff(Q_q, s) - (2*A_q*s + B_q)) == 0
assert expand(((aL - 3)*s + 4*(1 - aL)).subs(s, Rational(4, 3)) + Rational(8, 3)*aL) == 0
print("Monotonicity: OK")

print("All checks passed")

Setup: OK
Bias derivatives: OK
B_DD < 0: OK
B_TSRN > 0: OK
Limits of Psi: OK
Unique threshold: OK
a* at alpha_L = 1/2: 8*sqrt(7)/21 + 4/3 = 2.341238594691273
Monotonicity: OK
All checks passed
